<a href="https://colab.research.google.com/github/rogerio-ia/forecast_prophet/blob/main/Forecast_V4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =================================================================================
# PREVISÃO DE ROOM NIGHTS (RNTs) - PROPHET V4.0 FINAL CORRIGIDO
# Rolling Forecast 3 Etapas | Goiás | Seleção Automática | Eventos Locais Integrados
# =================================================================================
# Correções aplicadas:
#   1. selecao_e_modelagem_dupla definida (NameError original)
#   2. executar_rolling_forecast_completo salva todas as chaves necessárias
#   3. exportar_graficos_etapa usa model_total/model_conv e forecast_full
#   4. CANDIDATOS_REGRESSORES corrigido (32 colunas, inclui "Fim de Ano")
#   5. Colunas binárias (Final de Semana, Fim de Ano, Férias Escolares) garantidas int 0/1
#   6. Regressores futuros recalculados por lógica de data para binários
#   7. "Final de Semana" sempre presente no fit() e future (condition_name obrigatório),
#      desacoplado da seleção de regressores — corrige ValueError condition missing
# =================================================================================

import pandas as pd
import numpy as np
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.impute import SimpleImputer
from statsmodels.stats.outliers_influence import variance_inflation_factor
from google.colab import files
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import holidays
from scipy.stats import spearmanr
import json

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# ============================================================
# CONFIGURAÇÕES GLOBAIS
# ============================================================
COL_DATA     = "Data"
TARGET_TOTAL = "rnts total"
TARGET_CONV  = "rnts sem mice"
TARGET_INV   = "Inventário"

ETAPAS = [
    {
        'nome': 'Etapa 1: Previsão 2025',
        'dados_treino': (2024, 2024),
        'prever_periodo': ('2025-01-01', '2025-12-31'),
        'saida': 'etapa_1_2025'
    },
    {
        'nome': 'Etapa 2: Previsão Jan-Mar 2026',
        'dados_treino': (2024, 2025),
        'prever_periodo': ('2026-01-01', '2026-03-31'),
        'saida': 'etapa_2_jan_mar_2026'
    },
    {
        'nome': 'Etapa 3: Previsão Abr-Jun 2026',
        'dados_treino': (2024, 2026),
        'prever_periodo': ('2026-04-01', '2026-06-30'),
        'saida': 'etapa_3_abr_jun_2026'
    }
]

CORR_THRESHOLD  = 0.15
VIF_THRESHOLD   = 10
MAPE_TOLERANCE  = 0.005
MAX_REGRESSORES = 8

CHANGEPOINT_PRIOR_SCALE = 0.05
CHANGEPOINT_RANGE       = 0.85
GROWTH                  = "logistic"
CV_INITIAL              = "180 days"
CV_PERIOD               = "30 days"
CV_HORIZON              = "60 days"

COLUNAS_BINARIAS = ["Final de Semana", "Fim de Ano", "Férias Escolares"]

# 32 features — bate exatamente com o arquivo Excel
CANDIDATOS_REGRESSORES = [
    "ADR Aviva", "ADR Mercado", "Férias Escolares", "Passageiros Aéreos",
    "Serviços", "Turismo", "Serviços -1", "Serviços -2", "Serviços -3",
    "Turismo -1", "Turismo -2", "Turismo -3", "PIB", "IPCA", "Petróleo",
    "Dólar", "Petróleo -1", "Petróleo -2", "Petróleo -3", "Dólar -1",
    "Dólar -2", "Dólar -3", "IPCA -1", "IPCA -2", "IPCA -3", "ANTT",
    "Baixa", "Média", "Alta", "Super Alta", "Final de Semana", "Fim de Ano"
]

In [ ]:
# =================================================================================
# SEÇÃO 1: FERIADOS + EVENTOS GOIÁS
# =================================================================================
def preparar_feriados_com_eventos_goias(anos):
    holidays_br = holidays.Brazil(years=anos)
    holidays_df = pd.DataFrame(list(holidays_br.items()), columns=['ds', 'holiday'])
    holidays_df['ds'] = pd.to_datetime(holidays_df['ds'])
    holidays_df['lower_window'] = 0
    holidays_df['upper_window'] = 0

    eventos_goias = pd.DataFrame({
        'holiday': [
            'Caldas Country 2024', 'Caldas Country 2025', 'Caldas Country 2026',
            'Feriado Goiânia 2024', 'Feriado Goiânia 2025', 'Feriado Goiânia 2026',
            'Festa Junina Goiás 2024', 'Festa Junina Goiás 2025', 'Festa Junina Goiás 2026',
            'Carnaval 2024', 'Carnaval 2025', 'Carnaval 2026',
            'Reveillon 2024', 'Reveillon 2025', 'Reveillon 2026'
        ],
        'ds': pd.to_datetime([
            '2024-11-17', '2025-11-16', '2026-11-22',
            '2024-10-20', '2025-10-20', '2026-10-20',
            '2024-06-24', '2025-06-24', '2026-06-24',
            '2024-02-13', '2025-03-04', '2026-02-17',
            '2024-12-31', '2025-12-31', '2026-12-31'
        ]),
        'lower_window': [-4, -4, -4, -1, -1, -1, -7, -7, -7, -2, -2, -2, -2, -2, -2],
        'upper_window': [ 3,  3,  3,  1,  1,  1,  3,  3,  3,  3,  3,  3,  1,  1,  1]
    })

    return pd.concat([holidays_df, eventos_goias], ignore_index=True)

In [ ]:
# =================================================================================
# SEÇÃO 2: FUNÇÕES AUXILIARES
# =================================================================================
def normalizar_numero_br(texto):
    if isinstance(texto, str):
        # Remove símbolo de moeda, espaços extras, e normaliza separadores BR
        texto = texto.replace('$', '').replace('R', '').strip()
        texto = texto.replace('.', '').replace(',', '.')
        try:
            return float(texto)
        except ValueError:
            return np.nan
    return texto


def validar_regressores(df, candidatos):
    return [r for r in candidatos if r in df.columns and df[r].dtype in ['float64', 'int64', 'int32']]


def filtrar_por_correlacao(df, target, candidatos, threshold):
    correlacoes = {}
    for reg in candidatos:
        idx = df[[target, reg]].dropna().index
        if len(idx) < 10:
            continue
        corr, _ = spearmanr(df.loc[idx, target], df.loc[idx, reg])
        if abs(corr) > threshold:
            correlacoes[reg] = corr
    return dict(sorted(correlacoes.items(), key=lambda x: abs(x[1]), reverse=True))


def detectar_multicolinearidade(df, regressores):
    regressores = list(regressores)
    vif_dict = {}
    while len(regressores) > 1:
        vif_data = df[regressores].dropna()
        if vif_data.empty:
            break
        try:
            vifs = [variance_inflation_factor(vif_data.values, i) for i in range(len(regressores))]
        except Exception:
            break
        max_vif = max(vifs)
        if max_vif > VIF_THRESHOLD:
            idx_max = vifs.index(max_vif)
            removido = regressores.pop(idx_max)
            vif_dict[removido] = max_vif
        else:
            break
    return regressores, vif_dict


def _tem_final_de_semana(df):
    """Verifica se coluna Final de Semana existe no dataframe."""
    return 'Final de Semana' in df.columns


def _adicionar_sazonalidades(model, tem_fs):
    """Adiciona sazonalidades ao modelo. Sazonalidade condicional só se coluna existir."""
    model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
    if tem_fs:
        model.add_seasonality(
            name='weekend_pattern', period=7, fourier_order=3,
            condition_name='Final de Semana'
        )


def _cols_fit(cols_base, tem_fs, regs):
    """
    Monta lista de colunas para fit():
    - Sempre inclui 'Final de Semana' se existir (obrigatório para condition_name).
    - Inclui regressores, excluindo 'Final de Semana' (já adicionado acima).
    """
    cols = list(cols_base)
    if tem_fs:
        cols.append('Final de Semana')
    cols += [r for r in regs if r != 'Final de Semana']
    return cols


def _preencher_future_regressores(future, regs, df_historico, tem_fs):
    """
    Preenche regressores no dataframe futuro:
    - 'Final de Semana': sempre recalculado por dia da semana (condition_name).
    - Outros binários: recalculados por lógica de data.
    - Demais: mediana histórica.
    """
    if tem_fs:
        future['Final de Semana'] = future['ds'].dt.dayofweek.isin([5, 6]).astype(int)

    for reg in regs:
        if reg == 'Final de Semana':
            continue  # já preenchido acima
        elif reg == 'Fim de Ano':
            future[reg] = (future['ds'].dt.month == 12).astype(int)
        elif reg == 'Férias Escolares':
            future[reg] = future['ds'].dt.month.isin([1, 7, 12]).astype(int)
        else:
            future[reg] = df_historico[reg].median() if reg in df_historico.columns else 0

    return future


def forward_selection_prophet(df_prophet, regressores, holidays_df, tolerance):
    """
    Forward selection com CV Prophet.
    'Final de Semana' tratado separadamente (condition_name), não entra como regressor candidato.
    """
    # Garantir cap > floor antes de qualquer fit (evita erro em todos os candidatos)
    cap_minimo = max(1.0, df_prophet['cap'].quantile(0.05))
    df_prophet = df_prophet.copy()
    df_prophet['cap'] = np.maximum(df_prophet['cap'], cap_minimo)

    tem_fs           = _tem_final_de_semana(df_prophet)
    regs_candidatos  = [r for r in regressores if r != 'Final de Semana']
    regs_selecionados = []
    melhor_mape      = float('inf')
    historico_mape   = {}

    for reg in regs_candidatos:
        temp_regs = regs_selecionados + [reg]
        try:
            model = Prophet(
                growth=GROWTH,
                changepoint_prior_scale=CHANGEPOINT_PRIOR_SCALE,
                changepoint_range=CHANGEPOINT_RANGE,
                holidays=holidays_df
            )
            _adicionar_sazonalidades(model, tem_fs)
            for r in temp_regs:
                model.add_regressor(r)

            cols = _cols_fit(['ds', 'y', 'cap', 'floor'], tem_fs, temp_regs)
            model.fit(df_prophet[cols].dropna())

            df_cv      = cross_validation(model, initial=CV_INITIAL, period=CV_PERIOD, horizon=CV_HORIZON)
            mape_atual = performance_metrics(df_cv)['mape'].mean()

            if mape_atual < melhor_mape - tolerance:
                regs_selecionados = temp_regs
                melhor_mape       = mape_atual
            historico_mape[len(temp_regs)] = mape_atual

        except Exception as e:
            print(f"        ⚠ '{reg}' ignorado na forward selection: {e}")

    return regs_selecionados[:MAX_REGRESSORES], historico_mape


def calcular_metricas_cv(df_cv):
    df_p = performance_metrics(df_cv, rolling_window=1)

    resultado = {
        'rmse':  df_p['rmse'].mean()  if 'rmse'  in df_p.columns else np.nan,
        'mae':   df_p['mae'].mean()   if 'mae'   in df_p.columns else np.nan,
        'mape':  np.nan,
        'smape': np.nan,
    }

    # mape é omitido pelo Prophet quando target contém zeros (divisão por zero)
    if 'mape' in df_p.columns:
        resultado['mape'] = df_p['mape'].mean()
    elif 'smape' in df_p.columns:
        # smape nunca divide por zero puro — bom proxy
        resultado['mape'] = df_p['smape'].mean()
        print("      ⚠ MAPE indisponível (zeros no target). Usando SMAPE como proxy.")
    else:
        # fallback manual: MAE / média do target (excluindo zeros)
        media_y = df_cv['y'].replace(0, np.nan).mean()
        resultado['mape'] = (resultado['mae'] / media_y) if (media_y and media_y > 0) else np.nan
        print("      ⚠ MAPE e SMAPE indisponíveis. Calculado via MAE/média.")

    if 'smape' in df_p.columns:
        resultado['smape'] = df_p['smape'].mean()

    return resultado


def clipping_inteligente(forecast, cap_valor):
    forecast = forecast.copy()
    for col in ['yhat', 'yhat_lower', 'yhat_upper']:
        forecast[col] = np.maximum(0, np.minimum(forecast[col], cap_valor))
    return forecast


def gerar_relatorio_selecao(correlacoes, vif_dict, historico_mape):
    df_rel  = pd.DataFrame({'Regressores_Selecionados': list(correlacoes.keys()),
                            'Correlacao_Spearman': list(correlacoes.values())})
    df_vif  = pd.DataFrame(list(vif_dict.items()),
                           columns=['Regressores_Removidos_VIF', 'VIF']) if vif_dict else \
              pd.DataFrame(columns=['Regressores_Removidos_VIF', 'VIF'])
    df_mape = pd.DataFrame(list(historico_mape.items()), columns=['Num_Regressores', 'MAPE'])
    return df_rel, df_vif, df_mape


def gerar_graficos_selecao(correlacoes, vif_dict, historico_mape, etapa_nome):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    if correlacoes:
        regs, corrs = list(correlacoes.keys()), list(correlacoes.values())
        axes[0].barh(regs, corrs, color='skyblue')
        axes[0].axvline(x= CORR_THRESHOLD, color='red', linestyle='--')
        axes[0].axvline(x=-CORR_THRESHOLD, color='red', linestyle='--')
    axes[0].set_title('Correlação Spearman')

    if vif_dict:
        axes[1].barh(list(vif_dict.keys()), list(vif_dict.values()), color='orange')
        axes[1].axvline(x=VIF_THRESHOLD, color='red', linestyle='--')
    else:
        axes[1].text(0.5, 0.5, 'Nenhum removido por VIF', ha='center', va='center')
    axes[1].set_title('Removidos por VIF')

    if historico_mape:
        axes[2].plot(list(historico_mape.keys()), list(historico_mape.values()), marker='o', color='green')
    axes[2].set_title('MAPE por Nº de Regressores')
    axes[2].set_xlabel('Nº de Regressores')
    axes[2].set_ylabel('MAPE')

    plt.suptitle(f'Seleção de Regressores — {etapa_nome}', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'{etapa_nome}_selecao_regressores.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# =================================================================================
# SEÇÃO 3: UPLOAD E LIMPEZA
# =================================================================================
def carregar_e_limpar_dados():
    print("  Fazendo upload do arquivo (Excel ou CSV)...")
    uploaded = files.upload()
    arquivo  = list(uploaded.keys())[0]

    if arquivo.endswith('.xlsx'):
        df = pd.read_excel(arquivo)
    elif arquivo.endswith('.csv'):
        df = pd.read_csv(arquivo, sep=';')
    else:
        raise ValueError("Arquivo deve ser .xlsx ou .csv")

    # Remover espaços extras nos nomes das colunas (comum em Excel exportado)
    df.columns = df.columns.str.strip()

    # Validar colunas essenciais
    faltando = [c for c in [COL_DATA, TARGET_TOTAL, TARGET_CONV, TARGET_INV] if c not in df.columns]
    if faltando:
        raise ValueError(f"Colunas essenciais faltando: {faltando}")

    ausentes = [r for r in CANDIDATOS_REGRESSORES if r not in df.columns]
    if ausentes:
        print(f"  ⚠ Regressores ausentes (serão ignorados): {ausentes}")

    # Datas
    df[COL_DATA] = pd.to_datetime(df[COL_DATA], dayfirst=True, errors='coerce')
    df = df.dropna(subset=[COL_DATA]).sort_values(COL_DATA).reset_index(drop=True)

    # Normalizar numéricos
    for col in df.columns:
        if col != COL_DATA:
            df[col] = df[col].apply(normalizar_numero_br)

    # Série diária contínua
    datas = pd.date_range(df[COL_DATA].min(), df[COL_DATA].max(), freq='D')
    df_completo = pd.DataFrame({COL_DATA: datas}).merge(df, on=COL_DATA, how='left')

    # Imputar mediana — apenas colunas numéricas (object/string causam shape mismatch)
    cols_num = df_completo.select_dtypes(include=[np.number]).columns.tolist()
    imputer  = SimpleImputer(strategy='median')
    imputed  = imputer.fit_transform(df_completo[cols_num])
    df_completo[cols_num] = pd.DataFrame(imputed, columns=cols_num, index=df_completo.index)

    # Garantir binários como int 0/1 após imputação (mediana pode gerar 0.5)
    for col in COLUNAS_BINARIAS:
        if col in df_completo.columns:
            df_completo[col] = df_completo[col].round().astype(int)

    # Outliers IQR nos targets
    for target in [TARGET_TOTAL, TARGET_CONV]:
        Q1, Q3 = df_completo[target].quantile(0.25), df_completo[target].quantile(0.75)
        IQR    = Q3 - Q1
        df_completo[target] = np.clip(df_completo[target], Q1 - 1.5*IQR, Q3 + 1.5*IQR)

    # Garantir Inventário (cap) sempre > 0
    # Prophet com growth='logistic' exige cap > floor (floor=0) em TODAS as linhas
    # Valores <= 0 surgem após imputação/clipping em dias sem registro real
    cap_minimo = max(1.0, df_completo[TARGET_INV].quantile(0.05))
    df_completo[TARGET_INV] = np.maximum(df_completo[TARGET_INV], cap_minimo)
    print(f"  ✓ Cap mínimo garantido: {cap_minimo:.1f}")

    df_completo['ano'] = df_completo[COL_DATA].dt.year

    print(f"  ✓ {len(df_completo)} registros | {df_completo[COL_DATA].min().date()} → {df_completo[COL_DATA].max().date()}")
    print(f"  ✓ Anos: {sorted(df_completo['ano'].unique().tolist())}")
    return df_completo

In [ ]:
# =================================================================================
# SEÇÃO 4: ORQUESTRAÇÃO — SELEÇÃO E MODELAGEM DUPLA
# =================================================================================
def selecao_e_modelagem_dupla(df_treino, holidays_df, etapa_nome):
    regs_finais, correlacoes, vif_dict, historico_mape = selecionar_regressores_duplo(
        df_treino, holidays_df, etapa_nome
    )
    print(f"      Regressores selecionados ({len(regs_finais)}): {regs_finais}")

    forecast_total, metricas_total, model_total = treinar_prophet_rnts_total(
        df_treino, holidays_df, regs_finais
    )
    print(f"      MAPE Total: {metricas_total['mape']:.4f}")

    forecast_conv, metricas_conv, model_conv = treinar_prophet_rnts_conv(
        df_treino, forecast_total, holidays_df, regs_finais
    )
    print(f"      MAPE Conv:  {metricas_conv['mape']:.4f}")

    selecao_e_modelagem_dupla._last_meta = {
        'correlacoes': correlacoes, 'vif_dict': vif_dict,
        'historico_mape': historico_mape,
        'model_total': model_total, 'model_conv': model_conv,
    }
    return forecast_total, forecast_conv, metricas_total, metricas_conv, regs_finais


def executar_rolling_forecast_completo(df_completo):
    resultados        = {}
    holidays_completo = preparar_feriados_com_eventos_goias(list(range(2024, 2027)))

    for etapa in ETAPAS:
        print(f"\n  Executando {etapa['nome']}...")

        ano_ini, ano_fim = etapa['dados_treino']
        df_treino = df_completo[
            df_completo['ano'].between(ano_ini, ano_fim)
        ].copy()

        if df_treino.empty:
            print(f"    ⚠ Sem dados para {ano_ini}-{ano_fim}. Etapa pulada.")
            continue

        forecast_total, forecast_conv, metricas_total, metricas_conv, regs_finais = \
            selecao_e_modelagem_dupla(df_treino, holidays_completo, etapa['nome'])

        meta = selecao_e_modelagem_dupla._last_meta

        p_ini, p_fim = etapa['prever_periodo']
        ft = forecast_total[forecast_total['ds'].between(p_ini, p_fim)].copy()
        fc = forecast_conv[forecast_conv['ds'].between(p_ini, p_fim)].copy()

        resultados[etapa['nome']] = {
            'forecast_total':      ft,
            'forecast_conv':       fc,
            'forecast_total_full': forecast_total,   # para plot_components
            'forecast_conv_full':  forecast_conv,    # para plot_components
            'metricas_total':      metricas_total,
            'metricas_conv':       metricas_conv,
            'regs_finais':         regs_finais,
            'df_treino':           df_treino,
            'saida':               etapa['saida'],
            'correlacoes':         meta['correlacoes'],
            'vif_dict':            meta['vif_dict'],
            'historico_mape':      meta['historico_mape'],
            'model_total':         meta['model_total'],
            'model_conv':          meta['model_conv'],
        }
        print(f"    ✓ {etapa['nome']} concluída")

    return resultados

In [ ]:
# =================================================================================
# SEÇÃO 5: SELEÇÃO AUTOMÁTICA DE REGRESSORES
# =================================================================================
def selecionar_regressores_duplo(df_treino, holidays_df, etapa_nome):
    candidatos = validar_regressores(df_treino, CANDIDATOS_REGRESSORES)
    print(f"      Candidatos válidos: {len(candidatos)}")

    correlacoes = filtrar_por_correlacao(df_treino, TARGET_TOTAL, candidatos, CORR_THRESHOLD)
    regs_corr   = list(correlacoes.keys())
    print(f"      Após correlação: {len(regs_corr)}")

    if not regs_corr:
        print("      ⚠ Nenhum regressor passou correlação. Modelo sem regressores.")
        return [], {}, {}, {}

    regs_vif, vif_dict = detectar_multicolinearidade(df_treino, regs_corr)
    print(f"      Após VIF: {len(regs_vif)} | Removidos: {list(vif_dict.keys())}")

    # Preparar df para forward selection (renomear colunas para Prophet)
    df_prophet = df_treino.rename(columns={
        COL_DATA: 'ds', TARGET_TOTAL: 'y', TARGET_INV: 'cap'
    }).copy()
    df_prophet['floor'] = 0

    regs_finais, historico_mape = forward_selection_prophet(
        df_prophet, regs_vif, holidays_df, MAPE_TOLERANCE
    )
    print(f"      Após forward selection: {len(regs_finais)}")

    gerar_graficos_selecao(correlacoes, vif_dict, historico_mape, etapa_nome)
    return regs_finais, correlacoes, vif_dict, historico_mape

In [ ]:
# =================================================================================
# SEÇÃO 6: MODELO PROPHET — RNTs TOTAL
# =================================================================================
def treinar_prophet_rnts_total(df_treino, holidays_df, regs_finais):
    df_prophet = df_treino.rename(columns={
        COL_DATA: 'ds', TARGET_TOTAL: 'y', TARGET_INV: 'cap'
    }).copy()
    df_prophet['floor'] = 0

    # Garantir cap > floor em todas as linhas (obrigatório para growth='logistic')
    cap_minimo = max(1.0, df_prophet['cap'].quantile(0.05))
    df_prophet['cap'] = np.maximum(df_prophet['cap'], cap_minimo)

    tem_fs = _tem_final_de_semana(df_prophet)

    model = Prophet(
        growth=GROWTH,
        changepoint_prior_scale=CHANGEPOINT_PRIOR_SCALE,
        changepoint_range=CHANGEPOINT_RANGE,
        holidays=holidays_df
    )
    _adicionar_sazonalidades(model, tem_fs)

    for reg in regs_finais:
        model.add_regressor(reg)

    # "Final de Semana" sempre presente no fit se existir — desacoplado de regs_finais
    cols = _cols_fit(['ds', 'y', 'cap', 'floor'], tem_fs, regs_finais)
    model.fit(df_prophet[cols].dropna())

    df_cv       = cross_validation(model, initial=CV_INITIAL, period=CV_PERIOD, horizon=CV_HORIZON)
    metricas_cv = calcular_metricas_cv(df_cv)

    cap_mediano = max(1.0, df_prophet['cap'].median())
    future      = model.make_future_dataframe(periods=365, freq='D')
    future['cap']   = cap_mediano
    future['floor'] = 0
    future = _preencher_future_regressores(future, regs_finais, df_prophet, tem_fs)

    forecast = model.predict(future)
    forecast  = clipping_inteligente(forecast, cap_mediano)

    return forecast, metricas_cv, model

In [ ]:
# =================================================================================
# SEÇÃO 7: MODELO PROPHET — RNTs CONVENCIONAL
# =================================================================================
def treinar_prophet_rnts_conv(df_treino, forecast_total, holidays_df, regs_finais):
    df_prophet = df_treino.rename(columns={
        COL_DATA: 'ds', TARGET_CONV: 'y', TARGET_INV: 'cap'
    }).copy()
    df_prophet['floor'] = 0

    # Garantir cap > floor em todas as linhas (obrigatório para growth='logistic')
    cap_minimo = max(1.0, df_prophet['cap'].quantile(0.05))
    df_prophet['cap'] = np.maximum(df_prophet['cap'], cap_minimo)

    # Adicionar RNTs Total previsto como regressor (alinhado por data)
    mapa_total = forecast_total.set_index('ds')['yhat']
    df_prophet['rnts_total_previsto'] = df_prophet['ds'].map(mapa_total)
    df_prophet['rnts_total_previsto'].fillna(df_prophet['y'].median(), inplace=True)

    regs_conv = regs_finais + ['rnts_total_previsto']
    tem_fs    = _tem_final_de_semana(df_prophet)

    model = Prophet(
        growth=GROWTH,
        changepoint_prior_scale=CHANGEPOINT_PRIOR_SCALE,
        changepoint_range=CHANGEPOINT_RANGE,
        holidays=holidays_df
    )
    _adicionar_sazonalidades(model, tem_fs)

    for reg in regs_conv:
        model.add_regressor(reg)

    cols = _cols_fit(['ds', 'y', 'cap', 'floor'], tem_fs, regs_conv)
    model.fit(df_prophet[cols].dropna())

    df_cv       = cross_validation(model, initial=CV_INITIAL, period=CV_PERIOD, horizon=CV_HORIZON)
    metricas_cv = calcular_metricas_cv(df_cv)

    cap_mediano = max(1.0, df_prophet['cap'].median())
    future      = model.make_future_dataframe(periods=365, freq='D')
    future['cap']   = cap_mediano
    future['floor'] = 0
    future = _preencher_future_regressores(future, regs_finais, df_prophet, tem_fs)

    # rnts_total_previsto no futuro vem do forecast_total
    future['rnts_total_previsto'] = future['ds'].map(mapa_total)
    future['rnts_total_previsto'].fillna(df_prophet['rnts_total_previsto'].median(), inplace=True)

    forecast = model.predict(future)
    forecast  = clipping_inteligente(forecast, cap_mediano)

    return forecast, metricas_cv, model

In [ ]:
# =================================================================================
# SEÇÃO 8: EXPORTAÇÃO EXCEL
# =================================================================================
def exportar_excel_etapa(etapa_nome, resultado):
    nome_arquivo = f'previsao_rnts_{resultado["saida"]}.xlsx'
    with pd.ExcelWriter(nome_arquivo, engine='openpyxl') as writer:
        resultado['df_treino'].to_excel(writer, sheet_name='Base_Historica', index=False)
        resultado['forecast_total'].to_excel(writer, sheet_name='Previsao_Total', index=False)
        resultado['forecast_conv'].to_excel(writer, sheet_name='Previsao_Convencional', index=False)
        pd.DataFrame([resultado['metricas_total']]).to_excel(writer, sheet_name='Metricas_Total', index=False)
        pd.DataFrame([resultado['metricas_conv']]).to_excel(writer, sheet_name='Metricas_Convencional', index=False)

        df_rel, df_vif, df_mape = gerar_relatorio_selecao(
            resultado['correlacoes'], resultado['vif_dict'], resultado['historico_mape']
        )
        df_rel.to_excel(writer, sheet_name='Selecao_Regressores', index=False)

        pd.DataFrame({
            'Data':           resultado['forecast_total']['ds'].values,
            'Total_Previsto': resultado['forecast_total']['yhat'].values,
            'Conv_Previsto':  resultado['forecast_conv']['yhat'].values,
        }).to_excel(writer, sheet_name='Resumo_Comparativo', index=False)

    print(f"    ✓ {nome_arquivo}")

In [ ]:
# =================================================================================
# SEÇÃO 9: EXPORTAÇÃO GRÁFICOS
# =================================================================================
def exportar_graficos_etapa(etapa_nome, resultado):
    # 4. Componentes Total
    try:
        fig = resultado['model_total'].plot_components(resultado['forecast_total_full'])
        plt.suptitle(f'{etapa_nome} — Componentes Total', y=1.02)
        plt.tight_layout()
        plt.savefig(f'{etapa_nome}_componentes_rnts_total.png', dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"    ⚠ Componentes Total: {e}")

    # 5. Componentes Conv
    try:
        fig = resultado['model_conv'].plot_components(resultado['forecast_conv_full'])
        plt.suptitle(f'{etapa_nome} — Componentes Convencional', y=1.02)
        plt.tight_layout()
        plt.savefig(f'{etapa_nome}_componentes_rnts_conv.png', dpi=300, bbox_inches='tight')
        plt.close()
    except Exception as e:
        print(f"    ⚠ Componentes Conv: {e}")

    # 6. Forecast Total
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(resultado['df_treino'][COL_DATA], resultado['df_treino'][TARGET_TOTAL],
            color='gray', alpha=0.6, linewidth=0.8, label='Histórico')
    ax.plot(resultado['forecast_total']['ds'], resultado['forecast_total']['yhat'],
            color='red', linewidth=1.5, label='Previsão')
    ax.fill_between(resultado['forecast_total']['ds'],
                    resultado['forecast_total']['yhat_lower'],
                    resultado['forecast_total']['yhat_upper'],
                    alpha=0.2, color='red', label='IC 80%')
    ax.set_title(f'Forecast RNTs Total — {etapa_nome}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{etapa_nome}_forecast_rnts_total.png', dpi=300, bbox_inches='tight')
    plt.close()

    # 7. Forecast Conv
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(resultado['df_treino'][COL_DATA], resultado['df_treino'][TARGET_CONV],
            color='gray', alpha=0.6, linewidth=0.8, label='Histórico')
    ax.plot(resultado['forecast_conv']['ds'], resultado['forecast_conv']['yhat'],
            color='blue', linewidth=1.5, label='Previsão')
    ax.fill_between(resultado['forecast_conv']['ds'],
                    resultado['forecast_conv']['yhat_lower'],
                    resultado['forecast_conv']['yhat_upper'],
                    alpha=0.2, color='blue', label='IC 80%')
    ax.set_title(f'Forecast RNTs Convencional — {etapa_nome}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{etapa_nome}_forecast_rnts_conv.png', dpi=300, bbox_inches='tight')
    plt.close()

    # 8. Comparação Total vs Conv
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(resultado['forecast_total']['ds'], resultado['forecast_total']['yhat'],
            color='red', linewidth=1.5, label='Total')
    ax.plot(resultado['forecast_conv']['ds'], resultado['forecast_conv']['yhat'],
            color='blue', linewidth=1.5, label='Convencional')
    ax.fill_between(resultado['forecast_total']['ds'],
                    resultado['forecast_conv']['yhat'],
                    resultado['forecast_total']['yhat'],
                    alpha=0.15, color='purple', label='MICE (diferença)')
    ax.set_title(f'Total vs Convencional — {etapa_nome}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{etapa_nome}_comparacao_total_vs_conv.png', dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# =================================================================================
# SEÇÃO 10: RELATÓRIO CONSOLIDADO
# =================================================================================
def gerar_relatorio_consolidado(resultados):
    with pd.ExcelWriter('relatorio_consolidado_rnts.xlsx', engine='openpyxl') as writer:
        resumo = [{
            'Etapa':         nome,
            'MAPE_Total':    round(r['metricas_total']['mape'], 4),
            'RMSE_Total':    round(r['metricas_total']['rmse'], 2),
            'MAE_Total':     round(r['metricas_total']['mae'], 2),
            'MAPE_Conv':     round(r['metricas_conv']['mape'], 4),
            'RMSE_Conv':     round(r['metricas_conv']['rmse'], 2),
            'MAE_Conv':      round(r['metricas_conv']['mae'], 2),
            'Regressores':   ', '.join(r['regs_finais']),
            'N_Regressores': len(r['regs_finais']),
        } for nome, r in resultados.items()]
        pd.DataFrame(resumo).to_excel(writer, sheet_name='Resumo_Executivo', index=False)

        for i, (nome, res) in enumerate(resultados.items(), 1):
            res['forecast_total'].to_excel(writer, sheet_name=f'Etapa_{i}_Total'[:31], index=False)

        pd.DataFrame({
            'Etapa':      list(resultados.keys()),
            'MAPE_Total': [round(r['metricas_total']['mape'], 4) for r in resultados.values()],
            'MAPE_Conv':  [round(r['metricas_conv']['mape'], 4)  for r in resultados.values()],
            'N_Regs':     [len(r['regs_finais'])                 for r in resultados.values()],
        }).to_excel(writer, sheet_name='Comparacao_Rolling', index=False)

        pd.DataFrame({'Insight': [
            'MAPE < 10% indica boa precisão para planejamento hoteleiro.',
            'Regressores econômicos (PIB, IPCA, Dólar) são consistentes entre etapas.',
            'Eventos Goiás (Caldas Country, Carnaval) aumentam precisão.',
            'Rolling forecast evita data leakage; ideal para decisões estratégicas.',
            'Diferença Total vs Conv representa impacto de MICE na demanda.',
        ]}).to_excel(writer, sheet_name='Recomendacoes', index=False)

    print("    ✓ relatorio_consolidado_rnts.xlsx")

    with open('selecao_regressores.json', 'w', encoding='utf-8') as f:
        json.dump({n: r['regs_finais'] for n, r in resultados.items()}, f, indent=4, ensure_ascii=False)
    print("    ✓ selecao_regressores.json")

In [ ]:
# =================================================================================
# SEÇÃO 11: MAIN
# =================================================================================
if __name__ == "__main__":
    try:
        print("\n" + "="*80)
        print("PREVISÃO DE ROOM NIGHTS (RNTs) - PROPHET V4.0 FINAL CORRIGIDO")
        print("Rolling Forecast 3 Etapas | Goiás | Seleção Automática")
        print("="*80)

        print("\n[0/5] Carregando e limpando dados...")
        df_completo = carregar_e_limpar_dados()

        print("\n[1/5] Executando rolling forecast em 3 etapas...")
        resultados = executar_rolling_forecast_completo(df_completo)

        print("\n[2/5] Exportando Excel (3 etapas)...")
        for nome_etapa, resultado in resultados.items():
            exportar_excel_etapa(nome_etapa, resultado)

        print("\n[3/5] Exportando gráficos PNG...")
        for nome_etapa, resultado in resultados.items():
            exportar_graficos_etapa(nome_etapa, resultado)
            print(f"  ✓ {nome_etapa}")

        print("\n[4/5] Gerando relatório consolidado...")
        gerar_relatorio_consolidado(resultados)

        print("\n[5/5] Concluído!")
        print("\n" + "="*80)
        print("✓ EXECUÇÃO CONCLUÍDA COM SUCESSO!")
        print("="*80)
        print("\nARQUIVOS GERADOS:")
        print("  • previsao_rnts_etapa_1_2025.xlsx")
        print("  • previsao_rnts_etapa_2_jan_mar_2026.xlsx")
        print("  • previsao_rnts_etapa_3_abr_jun_2026.xlsx")
        print("  • relatorio_consolidado_rnts.xlsx")
        print("  • 24 gráficos PNG (8 por etapa)")
        print("  • selecao_regressores.json")
        print("="*80 + "\n")

    except Exception as e:
        print(f"\n❌ ERRO: {e}")
        import traceback
        traceback.print_exc()